# 02 — Signal Research

Evaluate individual alpha signals: quintile analysis, IC plots, signal decay.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config
from src.data.loader import DataLoader
from src.data.cleaner import DataCleaner
from src.data.universe import UniverseProvider
from src.features.registry import build_features
from src.analytics.statistics import information_coefficient
from src.visualization.plots import plot_quintile_returns, plot_ic_timeseries

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Load data
cfg = load_config('../config.yaml')
universe = UniverseProvider(cfg.data)
tickers = universe.get_tickers()

loader = DataLoader(cfg.data)
raw_data = loader.load_universe(tickers)
prices = loader.build_price_matrix(raw_data)
volumes = loader.build_volume_matrix(raw_data)

cleaner = DataCleaner(cfg.data)
prices, returns = cleaner.clean(prices, volumes)

# Forward returns for IC
fwd_returns = returns.shift(-1)

In [ ]:
# Compute all features
features = build_features(cfg.features)
signal_dict = {}
for feature in features:
    print(f'Computing: {feature.name}')
    signal_dict[feature.name] = feature.compute(prices, returns)
print(f'\nComputed {len(signal_dict)} signals')

In [ ]:
# IC summary for each signal
ic_summary = []
for name, sig in signal_dict.items():
    ic = information_coefficient(sig, fwd_returns)
    ic_summary.append({
        'Signal': name,
        'Mean IC': ic.mean(),
        'Std IC': ic.std(),
        'IC IR': ic.mean() / ic.std() if ic.std() > 0 else 0,
        'Hit Rate': (ic > 0).mean(),
    })

ic_df = pd.DataFrame(ic_summary).sort_values('IC IR', ascending=False)
print(ic_df.to_string(index=False, float_format='%.4f'))

In [ ]:
# Quintile analysis for top signals
for name in list(signal_dict.keys())[:4]:
    sig = signal_dict[name]
    plot_quintile_returns(sig, fwd_returns, title=f'Quintile Returns — {name}')
    plt.show()

In [ ]:
# IC time series for key signals
for name in ['composite_momentum', 'bollinger_mr', 'short_term_reversal']:
    if name in signal_dict:
        ic = information_coefficient(signal_dict[name], fwd_returns)
        plot_ic_timeseries(ic, signal_name=name)
        plt.show()

In [ ]:
# Signal decay analysis: IC at different forward horizons
horizons = [1, 5, 10, 21, 63]
decay_data = []

for name in ['composite_momentum', 'bollinger_mr']:
    if name not in signal_dict:
        continue
    sig = signal_dict[name]
    for h in horizons:
        fwd_h = returns.rolling(h).sum().shift(-h)
        ic = information_coefficient(sig, fwd_h)
        decay_data.append({'Signal': name, 'Horizon': h, 'Mean IC': ic.mean()})

if decay_data:
    decay_df = pd.DataFrame(decay_data)
    fig, ax = plt.subplots(figsize=(10, 5))
    for name in decay_df['Signal'].unique():
        subset = decay_df[decay_df['Signal'] == name]
        ax.plot(subset['Horizon'], subset['Mean IC'], marker='o', label=name)
    ax.set_title('Signal IC Decay')
    ax.set_xlabel('Forward Horizon (days)')
    ax.set_ylabel('Mean IC')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Signal correlation matrix
# Check for redundancy between signals
signal_corr = pd.DataFrame()
for name, sig in signal_dict.items():
    # Use cross-sectional mean as time-series proxy
    signal_corr[name] = sig.mean(axis=1)

corr = signal_corr.dropna().corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(corr.columns)))
ax.set_yticklabels(corr.columns, fontsize=8)
ax.set_title('Signal Correlation Matrix')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.show()